# 03 - Feature Engineering

Objetivo: definir el set final de variables predictoras y construir un preprocesador reproducible para los modelos del notebook 04.

Principios:
- Usar solo `TRAIN_SET` para ajustar transformadores.
- Usar `VAL_SET` solo para validar que el pipeline transforma datos futuros.
- Mantener `TEST_SET` reservado para la evaluacion final del modelo ganador.
- Evitar columnas de trazabilidad o leakage.


## 1) Setup y conexion


In [14]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import snowflake.connector

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.utils.config import get_project_config

pd.set_option("display.max_columns", 160)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

cfg = get_project_config(required=True)
train_table = cfg.fq_table(cfg.train_table)
val_table = cfg.fq_table(cfg.val_table)
test_table = cfg.fq_table(cfg.test_table)
target = cfg.target_column

print("TRAIN:", train_table)
print("VAL:", val_table)
print("TEST:", test_table)
print("Target:", target)


TRAIN: ANALYTICS.TRAIN_SET
VAL: ANALYTICS.VAL_SET
TEST: ANALYTICS.TEST_SET
Target: TOTAL_AMOUNT


In [15]:
def get_connection():
    return snowflake.connector.connect(**cfg.snowflake.connector_kwargs)


def query_df(sql: str) -> pd.DataFrame:
    with get_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql)
            return cur.fetch_pandas_all()


## 2) Columnas finales

No se usan columnas de cierre del viaje, componentes del precio ni identificadores crudos. `TRIP_NK`, `PICKUP_DATETIME` y `PICKUP_DATE` quedan fuera del modelo.


In [16]:
NUMERIC_FEATURES = [
    "TRIP_DISTANCE",
    "PASSENGER_COUNT",
    "PICKUP_HOUR",
    "DAY_OF_WEEK",
    "MONTH",
    "YEAR",
    "IS_WEEKEND",
    "SAME_BOROUGH_FLAG",
    "AIRPORT_TRIP_FLAG",
]

LOW_CARDINALITY_CATEGORICAL_FEATURES = [
    "SERVICE_TYPE",
    "VENDOR_ID",
    "RATE_CODE_ID",
    "TRIP_TYPE",
    "PICKUP_TIME_BAND",
    "PU_BOROUGH",
    "DO_BOROUGH",
]

HIGH_CARDINALITY_BASE_FEATURES = [
    "PU_LOCATION_ID",
    "DO_LOCATION_ID",
    "LOCATION_PAIR",
]

TRACEABILITY_COLUMNS = ["TRIP_NK", "PICKUP_DATETIME", "PICKUP_DATE"]
MODEL_FEATURES = NUMERIC_FEATURES + LOW_CARDINALITY_CATEGORICAL_FEATURES + HIGH_CARDINALITY_BASE_FEATURES

print("numeric:", NUMERIC_FEATURES)
print("low cardinality categorical:", LOW_CARDINALITY_CATEGORICAL_FEATURES)
print("high cardinality base:", HIGH_CARDINALITY_BASE_FEATURES)
print("total features:", len(MODEL_FEATURES))


numeric: ['TRIP_DISTANCE', 'PASSENGER_COUNT', 'PICKUP_HOUR', 'DAY_OF_WEEK', 'MONTH', 'YEAR', 'IS_WEEKEND', 'SAME_BOROUGH_FLAG', 'AIRPORT_TRIP_FLAG']
low cardinality categorical: ['SERVICE_TYPE', 'VENDOR_ID', 'RATE_CODE_ID', 'TRIP_TYPE', 'PICKUP_TIME_BAND', 'PU_BOROUGH', 'DO_BOROUGH']
high cardinality base: ['PU_LOCATION_ID', 'DO_LOCATION_ID', 'LOCATION_PAIR']
total features: 19


In [17]:
existing_columns_df = query_df(f"""
SELECT COLUMN_NAME
FROM INFORMATION_SCHEMA.COLUMNS
WHERE TABLE_SCHEMA = '{cfg.analytics_schema.upper()}'
  AND TABLE_NAME = '{cfg.train_table.upper()}'
ORDER BY ORDINAL_POSITION
""")
existing_columns = set(existing_columns_df["COLUMN_NAME"])
missing_features = sorted(set(MODEL_FEATURES + [target]) - existing_columns)
if missing_features:
    raise ValueError(f"Faltan columnas en TRAIN_SET: {missing_features}")
print("OK: todas las columnas del modelo existen en TRAIN_SET.")


OK: todas las columnas del modelo existen en TRAIN_SET.


## 3) Cardinalidad y estrategia top-N

`LOCATION_PAIR` tiene cardinalidad alta. Para evitar una matriz enorme, agrupamos pares poco frecuentes como `OTHER`. Los IDs de pickup/dropoff se mantienen porque tienen cardinalidad controlada.


In [18]:
cardinality_df = query_df(f"""
SELECT 'SERVICE_TYPE' AS column_name, COUNT(DISTINCT SERVICE_TYPE) AS distinct_values FROM {train_table}
UNION ALL SELECT 'VENDOR_ID', COUNT(DISTINCT VENDOR_ID) FROM {train_table}
UNION ALL SELECT 'RATE_CODE_ID', COUNT(DISTINCT RATE_CODE_ID) FROM {train_table}
UNION ALL SELECT 'TRIP_TYPE', COUNT(DISTINCT TRIP_TYPE) FROM {train_table}
UNION ALL SELECT 'PU_BOROUGH', COUNT(DISTINCT PU_BOROUGH) FROM {train_table}
UNION ALL SELECT 'DO_BOROUGH', COUNT(DISTINCT DO_BOROUGH) FROM {train_table}
UNION ALL SELECT 'PU_LOCATION_ID', COUNT(DISTINCT PU_LOCATION_ID) FROM {train_table}
UNION ALL SELECT 'DO_LOCATION_ID', COUNT(DISTINCT DO_LOCATION_ID) FROM {train_table}
UNION ALL SELECT 'LOCATION_PAIR', COUNT(DISTINCT LOCATION_PAIR) FROM {train_table}
ORDER BY distinct_values DESC
""")
cardinality_df


,COLUMN_NAME,DISTINCT_VALUES
0,LOCATION_PAIR,63249
1,PU_LOCATION_ID,265
2,DO_LOCATION_ID,265
3,PU_BOROUGH,8
4,DO_BOROUGH,8
5,RATE_CODE_ID,7
6,VENDOR_ID,6
7,TRIP_TYPE,2
8,SERVICE_TYPE,2


In [19]:
TOP_N_LOCATION_PAIRS = 300

location_pair_top_df = query_df(f"""
SELECT LOCATION_PAIR, COUNT(*) AS rows_total
FROM {train_table}
GROUP BY 1
ORDER BY rows_total DESC
LIMIT {TOP_N_LOCATION_PAIRS}
""")

TOP_LOCATION_PAIRS = set(location_pair_top_df["LOCATION_PAIR"].astype(str))
location_pair_top_df.head(20)


,LOCATION_PAIR,ROWS_TOTAL
0,264_264,6985111
1,237_236,3957112
2,236_237,3380152
3,236_236,3082317
4,237_237,2928078
5,239_238,1808784
6,239_142,1790265
7,237_162,1717992
8,142_239,1701715
9,237_161,1696797


## 4) Muestras de train y validacion

Bajamos muestras pequenas para probar transformaciones localmente. El entrenamiento serio del notebook 04 debe trabajar por muestras representativas o batches.


In [20]:
TRAIN_SAMPLE_PCT = 0.005
VAL_SAMPLE_PCT = 0.02
TRAIN_LIMIT = 120_000
VAL_LIMIT = 60_000

select_cols = ", ".join(MODEL_FEATURES + [target])

train_sample_df = query_df(f"""
SELECT {select_cols}
FROM {train_table} SAMPLE BERNOULLI ({TRAIN_SAMPLE_PCT})
LIMIT {TRAIN_LIMIT}
""")

val_sample_df = query_df(f"""
SELECT {select_cols}
FROM {val_table} SAMPLE BERNOULLI ({VAL_SAMPLE_PCT})
LIMIT {VAL_LIMIT}
""")

print("train sample:", train_sample_df.shape)
print("val sample:", val_sample_df.shape)
train_sample_df.head()


train sample: (38156, 20)
val sample: (7965, 20)


,TRIP_DISTANCE,PASSENGER_COUNT,PICKUP_HOUR,DAY_OF_WEEK,MONTH,YEAR,IS_WEEKEND,SAME_BOROUGH_FLAG,AIRPORT_TRIP_FLAG,SERVICE_TYPE,VENDOR_ID,RATE_CODE_ID,TRIP_TYPE,PICKUP_TIME_BAND,PU_BOROUGH,DO_BOROUGH,PU_LOCATION_ID,DO_LOCATION_ID,LOCATION_PAIR,TOTAL_AMOUNT
0,1.1000,1.0000,9,5,12,2017,0,1,0,yellow,2,1.0000,NaN,morning_peak,Manhattan,Manhattan,236,75,236_75,6.3000
1,1.4000,1.0000,17,5,12,2017,0,1,0,yellow,1,1.0000,NaN,evening_peak,Unknown,Unknown,264,264,264_264,11.8600
2,6.2000,1.0000,13,3,12,2017,0,1,0,yellow,1,1.0000,NaN,midday,Manhattan,Manhattan,229,13,229_13,26.6000
3,8.5400,1.0000,20,7,12,2017,1,1,0,yellow,2,1.0000,NaN,evening_peak,Manhattan,Manhattan,68,244,68_244,30.3000
4,4.5300,1.0000,0,2,12,2017,0,1,0,yellow,2,1.0000,NaN,overnight,Manhattan,Manhattan,79,261,79_261,20.1600


In [21]:
train_sample_df[MODEL_FEATURES + [target]].isna().mean().sort_values(ascending=False).head(20)


TRIP_TYPE           0.9181
PASSENGER_COUNT     0.0085
RATE_CODE_ID        0.0085
TRIP_DISTANCE       0.0000
DAY_OF_WEEK         0.0000
PICKUP_HOUR         0.0000
IS_WEEKEND          0.0000
MONTH               0.0000
SAME_BOROUGH_FLAG   0.0000
AIRPORT_TRIP_FLAG   0.0000
SERVICE_TYPE        0.0000
YEAR                0.0000
VENDOR_ID           0.0000
PICKUP_TIME_BAND    0.0000
PU_BOROUGH          0.0000
DO_BOROUGH          0.0000
PU_LOCATION_ID      0.0000
DO_LOCATION_ID      0.0000
LOCATION_PAIR       0.0000
TOTAL_AMOUNT        0.0000
dtype: float64

## 5) Transformaciones reproducibles


In [22]:
def apply_feature_engineering(df: pd.DataFrame, top_location_pairs: set[str]) -> pd.DataFrame:
    out = df.copy()

    # Normalizar categoricas como texto para OneHotEncoder y para estabilidad entre train/val/test.
    categorical_cols = LOW_CARDINALITY_CATEGORICAL_FEATURES + HIGH_CARDINALITY_BASE_FEATURES
    for col in categorical_cols:
        out[col] = out[col].astype("string").fillna("Unknown")

    out["LOCATION_PAIR_TOP"] = np.where(
        out["LOCATION_PAIR"].isin(top_location_pairs),
        out["LOCATION_PAIR"],
        "OTHER",
    )

    # Usaremos la version top-N y no la columna original completa.
    out = out.drop(columns=["LOCATION_PAIR"])
    return out

ENGINEERED_NUMERIC_FEATURES = NUMERIC_FEATURES
ENGINEERED_CATEGORICAL_FEATURES = LOW_CARDINALITY_CATEGORICAL_FEATURES + [
    "PU_LOCATION_ID",
    "DO_LOCATION_ID",
    "LOCATION_PAIR_TOP",
]
ENGINEERED_FEATURES = ENGINEERED_NUMERIC_FEATURES + ENGINEERED_CATEGORICAL_FEATURES

train_fe_df = apply_feature_engineering(train_sample_df, TOP_LOCATION_PAIRS)
val_fe_df = apply_feature_engineering(val_sample_df, TOP_LOCATION_PAIRS)

print("engineered train:", train_fe_df.shape)
print("engineered val:", val_fe_df.shape)
train_fe_df[ENGINEERED_FEATURES].head()


engineered train: (38156, 20)
engineered val: (7965, 20)


,TRIP_DISTANCE,PASSENGER_COUNT,PICKUP_HOUR,DAY_OF_WEEK,MONTH,YEAR,IS_WEEKEND,SAME_BOROUGH_FLAG,AIRPORT_TRIP_FLAG,SERVICE_TYPE,VENDOR_ID,RATE_CODE_ID,TRIP_TYPE,PICKUP_TIME_BAND,PU_BOROUGH,DO_BOROUGH,PU_LOCATION_ID,DO_LOCATION_ID,LOCATION_PAIR_TOP
0,1.1000,1.0000,9,5,12,2017,0,1,0,yellow,2,1.0,Unknown,morning_peak,Manhattan,Manhattan,236,75,236_75
1,1.4000,1.0000,17,5,12,2017,0,1,0,yellow,1,1.0,Unknown,evening_peak,Unknown,Unknown,264,264,264_264
2,6.2000,1.0000,13,3,12,2017,0,1,0,yellow,1,1.0,Unknown,midday,Manhattan,Manhattan,229,13,OTHER
3,8.5400,1.0000,20,7,12,2017,1,1,0,yellow,2,1.0,Unknown,evening_peak,Manhattan,Manhattan,68,244,OTHER
4,4.5300,1.0000,0,2,12,2017,0,1,0,yellow,2,1.0,Unknown,overnight,Manhattan,Manhattan,79,261,OTHER


In [23]:
numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50, sparse_output=True)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, ENGINEERED_NUMERIC_FEATURES),
        ("cat", categorical_pipeline, ENGINEERED_CATEGORICAL_FEATURES),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

X_train_sample = train_fe_df[ENGINEERED_FEATURES]
y_train_sample = train_fe_df[target]
X_val_sample = val_fe_df[ENGINEERED_FEATURES]
y_val_sample = val_fe_df[target]

X_train_transformed = preprocessor.fit_transform(X_train_sample)
X_val_transformed = preprocessor.transform(X_val_sample)

print("X_train transformed:", X_train_transformed.shape)
print("X_val transformed:", X_val_transformed.shape)
print("y_train:", y_train_sample.shape)
print("y_val:", y_val_sample.shape)


X_train transformed: (38156, 297)
X_val transformed: (7965, 297)
y_train: (38156,)
y_val: (7965,)


In [24]:
feature_names = preprocessor.get_feature_names_out()
print("Numero de columnas transformadas:", len(feature_names))
print(feature_names[:30])


Numero de columnas transformadas: 297
['num__TRIP_DISTANCE' 'num__PASSENGER_COUNT' 'num__PICKUP_HOUR'
 'num__DAY_OF_WEEK' 'num__MONTH' 'num__YEAR' 'num__IS_WEEKEND'
 'num__SAME_BOROUGH_FLAG' 'num__AIRPORT_TRIP_FLAG'
 'cat__SERVICE_TYPE_green' 'cat__SERVICE_TYPE_yellow' 'cat__VENDOR_ID_1'
 'cat__VENDOR_ID_2' 'cat__VENDOR_ID_infrequent_sklearn'
 'cat__RATE_CODE_ID_1.0' 'cat__RATE_CODE_ID_2.0' 'cat__RATE_CODE_ID_3.0'
 'cat__RATE_CODE_ID_5.0' 'cat__RATE_CODE_ID_Unknown'
 'cat__RATE_CODE_ID_infrequent_sklearn' 'cat__TRIP_TYPE_1.0'
 'cat__TRIP_TYPE_2.0' 'cat__TRIP_TYPE_Unknown'
 'cat__PICKUP_TIME_BAND_evening_peak' 'cat__PICKUP_TIME_BAND_midday'
 'cat__PICKUP_TIME_BAND_morning_peak' 'cat__PICKUP_TIME_BAND_night'
 'cat__PICKUP_TIME_BAND_overnight' 'cat__PU_BOROUGH_Bronx'
 'cat__PU_BOROUGH_Brooklyn']


## 6) Validacion de estabilidad train/val


In [25]:
target_summary_df = pd.DataFrame({
    "split": ["train_sample", "val_sample"],
    "rows": [len(y_train_sample), len(y_val_sample)],
    "target_mean": [y_train_sample.mean(), y_val_sample.mean()],
    "target_p50": [y_train_sample.median(), y_val_sample.median()],
    "target_p90": [y_train_sample.quantile(0.90), y_val_sample.quantile(0.90)],
    "target_p99": [y_train_sample.quantile(0.99), y_val_sample.quantile(0.99)],
})
target_summary_df


,split,rows,target_mean,target_p50,target_p90,target_p99
0,train_sample,38156,17.5624,13.1000,32.1600,73.7000
1,val_sample,7965,28.7371,21.0000,58.3440,105.2300


In [26]:
unknown_pair_rate = (val_fe_df["LOCATION_PAIR_TOP"] == "OTHER").mean()
print(f"LOCATION_PAIR_TOP = OTHER en validacion: {unknown_pair_rate:.2%}")

if X_train_transformed.shape[1] != X_val_transformed.shape[1]:
    raise ValueError("Train y val no tienen la misma dimension transformada.")
print("OK: el preprocesador transforma train y val con la misma dimension.")


LOCATION_PAIR_TOP = OTHER en validacion: 63.88%
OK: el preprocesador transforma train y val con la misma dimension.


## 7) Artefactos conceptuales para `src/features`

Estos objetos se deben migrar luego a `src/features/build_features.py`:
- `MODEL_FEATURES`
- `apply_feature_engineering`
- `ENGINEERED_NUMERIC_FEATURES`
- `ENGINEERED_CATEGORICAL_FEATURES`
- `ColumnTransformer` con imputacion, escalado y OneHotEncoder

Para el notebook 04, el flujo recomendado es:
1. Extraer muestra/batches de `TRAIN_SET`.
2. Calcular top-N de `LOCATION_PAIR` solo con train.
3. Ajustar el preprocesador solo con train.
4. Comparar modelos en `VAL_SET`.
5. Reservar `TEST_SET` para el modelo ganador.
